# Capstone 3 — Data Analysis with Pandas

**Aura / ClickO healthcare prep**  
Input: `NSMES1988updated.csv` (from Capstone 2 — age in years, income in USD)

Two parts:
1. Working with Pandas (dtypes, categoricals, pivots on Health & Region)
2. Analyze and Cleanse (visits, demographics, distribution tables)


## Task A — Import libraries and load data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('NSMES1988updated.csv')
df = pd.read_csv(DATA)
print('shape:', df.shape)
df.head()


shape: (4406, 18)


,visits,nvisits,ovisits,novisits,emergency,hospital,health,chronic,adl,region,age,gender,married,school,income,employed,insurance,medicaid
0,5,0,0,0,0,1,average,2,normal,other,69.0,male,yes,6,28810.0,yes,yes,no
1,1,0,2,0,2,0,average,2,normal,other,74.0,female,yes,10,27478.0,no,yes,no
2,13,0,0,0,3,3,poor,4,limited,other,66.0,female,no,10,6532.0,no,no,yes
3,16,0,5,0,1,1,poor,2,limited,other,76.0,male,yes,3,6588.0,no,yes,no
4,3,0,0,0,0,0,average,2,limited,other,79.0,female,yes,6,6588.0,no,yes,no


## Task A — Identify data types

In [2]:
print(df.dtypes)
print()
print(df.info())


visits         int64
nvisits        int64
ovisits        int64
novisits       int64
emergency      int64
hospital       int64
health           str
chronic        int64
adl              str
region           str
age          float64
gender           str
married          str
school         int64
income       float64
employed         str
insurance        str
medicaid         str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 4406 entries, 0 to 4405
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   visits     4406 non-null   int64  
 1   nvisits    4406 non-null   int64  
 2   ovisits    4406 non-null   int64  
 3   novisits   4406 non-null   int64  
 4   emergency  4406 non-null   int64  
 5   hospital   4406 non-null   int64  
 6   health     4406 non-null   str    
 7   chronic    4406 non-null   int64  
 8   adl        4406 non-null   str    
 9   region     4406 non-null   str    
 10  age        4406 non-null  

## Task A — Identify categorical types

In [3]:
# factors / categorical-like columns from the data dictionary
cat_cols = ['health', 'adl', 'region', 'gender', 'married', 'employed', 'insurance', 'medicaid']
print('Categorical (factor) columns:')
for c in cat_cols:
    print(f'  {c}: {df[c].nunique()} unique -> {sorted(df[c].astype(str).unique().tolist())}')

# also treat as categorical in a working frame for pivots
work = df.copy()
for c in cat_cols:
    work[c] = work[c].astype('category')
print()
print(work[cat_cols].dtypes)


Categorical (factor) columns:
  health: 3 unique -> ['average', 'excellent', 'poor']
  adl: 2 unique -> ['limited', 'normal']
  region: 4 unique -> ['midwest', 'northeast', 'other', 'west']
  gender: 2 unique -> ['female', 'male']
  married: 2 unique -> ['no', 'yes']
  employed: 2 unique -> ['no', 'yes']
  insurance: 2 unique -> ['no', 'yes']
  medicaid: 2 unique -> ['no', 'yes']

health       category
adl          category
region       category
gender       category
married      category
employed     category
insurance    category
medicaid     category
dtype: object


## Task A — Detailed pivoting (include Health and Region)

In [4]:
# Health x Region counts
pivot_health_region = pd.crosstab(work['health'], work['region'], margins=True)
print('Counts: Health x Region')
print(pivot_health_region)
print()

# mean visits by Health and Region
pivot_visits = work.pivot_table(values='visits', index='health', columns='region', aggfunc='mean')
print('Mean physician visits by Health x Region')
print(pivot_visits.round(2))
print()

# mean income by Health and Region
pivot_income = work.pivot_table(values='income', index='health', columns='region', aggfunc='mean')
print('Mean income (USD) by Health x Region')
print(pivot_income.round(0))
print()

# chronic conditions by Health / Region
pivot_chronic = work.pivot_table(values='chronic', index='health', columns='region', aggfunc='mean')
print('Mean chronic conditions by Health x Region')
print(pivot_chronic.round(2))


Counts: Health x Region
region     midwest  northeast  other  west   All
health                                          
average        957        694   1237   621  3509
excellent       90         57    105    91   343
poor           110         86    272    86   554
All           1157        837   1614   798  4406

Mean physician visits by Health x Region
region     midwest  northeast  other  west
health                                    
average       5.28       5.69   5.09  6.50
excellent     3.44       4.02   3.14  3.37
poor          8.12      10.67   8.75  8.59

Mean income (USD) by Health x Region
region     midwest  northeast    other     west
health                                         
average    24984.0    27015.0  22011.0  31664.0
excellent  31054.0    33400.0  30025.0  37256.0
poor       21618.0    20659.0  16852.0  21119.0

Mean chronic conditions by Health x Region
region     midwest  northeast  other  west
health                                    
average       1.3

### Task A — Analysis report

- Dtypes mix numeric counts (visits, hospital, school, …) with string/factor fields (health, region, gender, …).
- Categoricals for Aura-style aggregation: health, adl, region, gender, married, employed, insurance, medicaid.
- Pivots on **Health** and **Region** show how utilization (visits) and income differ across self-reported health and geography.
- Poorer self-rated health often ties to higher visit/chronic burden — useful signal for Aura audience segmentation later.


## Task B — Analysis by visits, gender, marital status, school, income, employment, insurance, medicaid

In [5]:
visit_cols = ['visits', 'nvisits', 'ovisits', 'novisits', 'emergency', 'hospital']
print('Visit-type summary:')
print(work[visit_cols].describe().round(2))
print()

print('Gender counts:')
print(work['gender'].value_counts())
print()
print('Marital status:')
print(work['married'].value_counts())
print()
print('Employment:')
print(work['employed'].value_counts())
print()
print('Insurance:')
print(work['insurance'].value_counts())
print()
print('Medicaid:')
print(work['medicaid'].value_counts())
print()
print('School (years) describe:')
print(work['school'].describe())
print()
print('Income describe:')
print(work['income'].describe())


Visit-type summary:
        visits  nvisits  ovisits  novisits  emergency  hospital
count  4406.00  4406.00  4406.00   4406.00    4406.00   4406.00
mean      5.77     1.62     0.75      0.54       0.26      0.30
std       6.76     5.32     3.65      3.88       0.70      0.75
min       0.00     0.00     0.00      0.00       0.00      0.00
25%       1.00     0.00     0.00      0.00       0.00      0.00
50%       4.00     0.00     0.00      0.00       0.00      0.00
75%       8.00     1.00     0.00      0.00       0.00      0.00
max      89.00   104.00   141.00    155.00      12.00      8.00

Gender counts:
gender
female    2628
male      1778
Name: count, dtype: int64

Marital status:
married
yes    2406
no     2000
Name: count, dtype: int64

Employment:
employed
no     3951
yes     455
Name: count, dtype: int64

Insurance:
insurance
yes    3421
no      985
Name: count, dtype: int64

Medicaid:
medicaid
no     4004
yes     402
Name: count, dtype: int64

School (years) describe:
count    4

## Task B — Age and Gender distribution

In [6]:
# age groups for a readable distribution table
bins = [0, 30, 40, 50, 60, 70, 80, 120]
labels = ['<30', '30-39', '40-49', '50-59', '60-69', '70-79', '80+']
work['age_group'] = pd.cut(work['age'], bins=bins, labels=labels, right=False)

age_gender = pd.crosstab(work['age_group'], work['gender'], margins=True)
print('Age group x Gender counts')
print(age_gender)


Age group x Gender counts
gender     female  male   All
age_group                    
60-69         723   554  1277
70-79        1369   924  2293
80+           536   300   836
All          2628  1778  4406


## Task B — Health status by Gender

In [7]:
health_gender = pd.crosstab(work['health'], work['gender'], margins=True)
print(health_gender)
print()
print('Row % within health:')
print(pd.crosstab(work['health'], work['gender'], normalize='index').round(3))


gender     female  male   All
health                       
average      2093  1416  3509
excellent     193   150   343
poor          342   212   554
All          2628  1778  4406

Row % within health:
gender     female   male
health                  
average     0.596  0.404
excellent   0.563  0.437
poor        0.617  0.383


## Task B — Income distribution by Gender

In [8]:
income_by_gender = work.groupby('gender', observed=True)['income'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print(income_by_gender.round(2))


        count      mean   median       std      min       max
gender                                                       
female   2628  22493.48  14160.0  27218.31 -10125.0  548351.0
male     1778  29377.16  20574.0  31573.05 -10125.0  548351.0


## Task B — Regional income distribution

In [9]:
income_by_region = work.groupby('region', observed=True)['income'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print(income_by_region.round(2))


           count      mean   median       std      min       max
region                                                          
midwest     1157  25136.34  17875.0  31078.41      0.0  548351.0
northeast    837  26797.09  17413.0  31962.29      0.0  417596.0
other       1614  21662.84  14220.0  23234.74 -10125.0  360024.0
west         798  31165.05  20656.0  33148.57  -8180.0  242161.0


## Task B — Age-wise income analysis

In [10]:
income_by_age = work.groupby('age_group', observed=True)['income'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print(income_by_age.round(2))
print()
# also correlation age vs income
print('Pearson corr(age, income):', round(work['age'].corr(work['income']), 4))


           count      mean   median       std      min       max
age_group                                                       
60-69       1277  27849.14  19936.0  28075.40  -8180.0  360024.0
70-79       2293  25083.97  16502.0  31268.93 -10125.0  548351.0
80+          836  21847.53  13479.0  24536.56      0.0  230179.0

Pearson corr(age, income): -0.0731


## Export key tables for Capstone 4 / review

In [11]:
out_dir = Path('tables')
out_dir.mkdir(exist_ok=True)
age_gender.to_csv(out_dir / 'age_gender_distribution.csv')
health_gender.to_csv(out_dir / 'health_by_gender.csv')
income_by_gender.to_csv(out_dir / 'income_by_gender.csv')
income_by_region.to_csv(out_dir / 'income_by_region.csv')
income_by_age.to_csv(out_dir / 'income_by_age_group.csv')
pivot_health_region.to_csv(out_dir / 'pivot_health_region_counts.csv')
pivot_visits.to_csv(out_dir / 'pivot_mean_visits_health_region.csv')
print('wrote tables to', out_dir.resolve())
list(out_dir.iterdir())


wrote tables to /workspace/course/out/capstones/C3- Applied data science - Python - Pandas analysis/tables


[PosixPath('tables/income_by_gender.csv'),
 PosixPath('tables/pivot_health_region_counts.csv'),
 PosixPath('tables/health_by_gender.csv'),
 PosixPath('tables/income_by_region.csv'),
 PosixPath('tables/age_gender_distribution.csv'),
 PosixPath('tables/pivot_mean_visits_health_region.csv'),
 PosixPath('tables/income_by_age_group.csv')]

## Task B — Findings report

- **Visits:** Most people have low office/hospital counts; emergency and hospital stays are rarer — skewed utilization.
- **Demographics:** Gender, marriage, employment, insurance, and medicaid split the sample into clear segments for Aura-style targeting.
- **Age × Gender:** Distribution tables show where the sample concentrates by life stage and gender.
- **Health × Gender:** Self-rated health patterns can differ by gender — check the crosstab for which groups report poorer health.
- **Income × Gender / Region / Age:** Means and medians highlight economic differences across segments; age–income correlation summarizes the overall link.
- These pivots and distribution tables are the inputs Capstone 4 should visualize.
